---
## Exercises

### Exercise 1: S3 File Manager
Build a function `s3_sync(local_dir, bucket, prefix)` that:
- Uploads all CSV/Parquet files from a local directory to S3
- Skips files that already exist (check by key existence)
- Prints a summary: N files uploaded, M files skipped

### Exercise 2: Data Pipeline Logger
Extend the `lambda_handler` to:
- Log start/end time and row count to a DynamoDB table (simulate locally with a dict)
- Return whether the data quality check passed (no nulls in required columns)

### Exercise 3: Cost Estimator
Write a function that:
- Lists all objects in a bucket with their sizes
- Calculates estimated monthly cost at S3 Standard rates ($0.023/GB)
- Recommends objects older than 90 days for Glacier archival

---
## Key Takeaways

1. **IAM**: Always use least-privilege policies. Use roles for services, not users.
2. **S3**: The data lake for ML. Organize with `raw/`, `processed/`, `models/` prefixes.
3. **Parquet**: ~5x smaller than CSV, faster reads — use it for anything > 10MB.
4. **Lambda**: Great for orchestration and light processing, not for heavy training.
5. **Boto3**: `client` for raw API calls, `resource` for higher-level interactions.
6. **Error handling**: Always handle `ClientError` and `NoCredentialsError`.

In [ ]:
class S3Helper:
    """Reusable S3 helper for ML workflows."""
    
    def __init__(self, bucket: str, region: str = 'us-east-1'):
        self.bucket = bucket
        self.s3 = boto3.client('s3', region_name=region)
    
    def upload_df(self, df: pd.DataFrame, key: str, fmt: str = 'parquet') -> None:
        buffer = io.BytesIO()
        if fmt == 'parquet':
            df.to_parquet(buffer, index=False)
            content_type = 'application/octet-stream'
        else:
            buffer = io.StringIO()
            df.to_csv(buffer, index=False)
            buffer = io.BytesIO(buffer.getvalue().encode())
            content_type = 'text/csv'
        
        buffer.seek(0)
        self.s3.put_object(Bucket=self.bucket, Key=key, Body=buffer.read(), ContentType=content_type)
        print(f"✓ s3://{self.bucket}/{key}")
    
    def download_df(self, key: str) -> pd.DataFrame:
        response = self.s3.get_object(Bucket=self.bucket, Key=key)
        body = response['Body'].read()
        if key.endswith('.parquet'):
            return pd.read_parquet(io.BytesIO(body))
        return pd.read_csv(io.StringIO(body.decode('utf-8')))
    
    def list_keys(self, prefix: str = '') -> list:
        keys = []
        paginator = self.s3.get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=self.bucket, Prefix=prefix):
            for obj in page.get('Contents', []):
                keys.append(obj['Key'])
        return keys
    
    def copy(self, source_key: str, dest_key: str) -> None:
        self.s3.copy_object(
            Bucket=self.bucket,
            CopySource={'Bucket': self.bucket, 'Key': source_key},
            Key=dest_key
        )
        print(f"✓ Copied {source_key} → {dest_key}")

print("S3Helper class ready!")
print("\nUsage:")
print("  helper = S3Helper('my-ml-project')")
print("  helper.upload_df(df, 'data/features.parquet')")
print("  df = helper.download_df('data/features.parquet')")

---
## Section 6: Boto3 Patterns

### Session vs Client vs Resource
```python
# Session: wraps credentials and region (useful for multiple profiles)
session = boto3.Session(profile_name='ml-dev', region_name='us-east-1')

# Client: low-level API, returns dicts
s3_client = session.client('s3')
s3_client.list_buckets()  # returns {'Buckets': [...], 'ResponseMetadata': {...}}

# Resource: high-level OO interface
s3_resource = session.resource('s3')
bucket = s3_resource.Bucket('my-bucket')
for obj in bucket.objects.all():
    print(obj.key)
```

### Pagination
AWS APIs return at most 1000 items per call. Use paginators for large lists:
```python
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket='my-bucket', Prefix='data/'):
    for obj in page.get('Contents', []):
        print(obj['Key'])
```

### Error Handling
```python
from botocore.exceptions import ClientError, NoCredentialsError

try:
    s3.get_object(Bucket='my-bucket', Key='missing.csv')
except ClientError as e:
    error_code = e.response['Error']['Code']
    if error_code == 'NoSuchKey':
        print("File not found in S3")
    elif error_code == 'AccessDenied':
        print("No permission to read this file")
except NoCredentialsError:
    print("AWS credentials not configured")
```

In [ ]:
def lambda_handler(event, context):
    """
    Example: S3-triggered Lambda that preprocesses new data.
    In production, this deploys to AWS Lambda.
    Here, we test the logic locally.
    """
    # Parse S3 event
    s3_key = event.get('key', 'raw/data.csv')
    bucket = event.get('bucket', 'my-ml-project')
    
    print(f"Processing: s3://{bucket}/{s3_key}")
    
    # Simulate loading from S3
    data = pd.DataFrame({
        'age': [25, 35, np.nan, 45],
        'income': [50000, np.nan, 60000, 70000],
    })
    
    # Preprocessing logic
    data_clean = data.fillna(data.mean())
    
    # Write processed back to S3
    output_key = s3_key.replace('raw/', 'processed/')
    print(f"Output: s3://{bucket}/{output_key}")
    
    return {
        'statusCode': 200,
        'rowsProcessed': len(data_clean),
        'outputKey': output_key
    }

# Test locally
event = {'bucket': 'my-ml-project', 'key': 'raw/features.csv'}
result = lambda_handler(event, None)
print(f"\nResult: {result}")

---
## Section 5: AWS Lambda for ML Workflows

Lambda is serverless compute — you write a function, AWS runs it.

### ML Use Cases
1. **Data ingestion trigger**: S3 upload → Lambda → trigger SageMaker job
2. **Lightweight inference**: Simple models that run in < 15 minutes
3. **Alerting**: CloudWatch metric → Lambda → Slack notification
4. **Data preprocessing**: Event-driven ETL

### Lambda Anatomy
```python
def handler(event, context):
    # event: what triggered this function (e.g., S3 event, API call)
    # context: runtime info (memory, timeout, request ID)
    
    # Your logic here
    result = process(event)
    
    return {
        'statusCode': 200,
        'body': json.dumps(result)
    }
```

### Limitations for ML
- Max 15 minutes execution time
- Max 10 GB memory
- No GPU
- Cold start latency (~100ms-1s)
→ Use Lambda for orchestration, not heavy training

In [ ]:
import io
import time

# Generate larger dataset
big_df = pd.DataFrame({
    'user_id': range(100_000),
    'age': np.random.randint(18, 80, 100_000),
    'income': np.random.normal(50000, 20000, 100_000),
    'category': np.random.choice(['A', 'B', 'C', 'D'], 100_000),
    'score': np.random.rand(100_000),
})

# CSV
csv_buffer = io.StringIO()
big_df.to_csv(csv_buffer, index=False)
csv_size = len(csv_buffer.getvalue().encode('utf-8'))

# Parquet
parquet_buffer = io.BytesIO()
big_df.to_parquet(parquet_buffer, index=False)
parquet_size = len(parquet_buffer.getvalue())

print(f"CSV size:     {csv_size / 1024 / 1024:.2f} MB")
print(f"Parquet size: {parquet_size / 1024 / 1024:.2f} MB")
print(f"Savings:      {(1 - parquet_size/csv_size)*100:.1f}%")

---
## Section 4: S3 Best Practices for ML

### Versioning
Enable S3 versioning to keep history of dataset changes:
```python
s3.put_bucket_versioning(
    Bucket=bucket_name,
    VersioningConfiguration={'Status': 'Enabled'}
)
```

### Lifecycle Policies
Automatically move old data to cheaper storage:
- After 30 days: Standard → Standard-IA
- After 90 days: Standard-IA → Glacier

### Cost Awareness
- S3 Standard: ~$0.023/GB/month
- Glacier: ~$0.004/GB/month
- Data transfer OUT: $0.09/GB (big cost in ML!)
- Tip: Keep training data in the SAME region as SageMaker

### Parquet vs CSV
For large datasets, always prefer Parquet:
- ~4-8x smaller than CSV
- Column-oriented (fast for feature selection)
- Preserves dtypes

In [ ]:
# Even without AWS, we can simulate the pattern locally
import tempfile
import shutil

class LocalS3Simulator:
    """Simulate S3 operations locally for practice."""
    
    def __init__(self, base_dir='/tmp/local-s3'):
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)
    
    def put(self, bucket, key, data):
        """Upload data to 'S3'."""
        path = self.base_dir / bucket / key
        path.parent.mkdir(parents=True, exist_ok=True)
        if isinstance(data, pd.DataFrame):
            data.to_csv(path, index=False)
        elif isinstance(data, str):
            path.write_text(data)
        elif isinstance(data, bytes):
            path.write_bytes(data)
        print(f"s3://{bucket}/{key} ✓")
    
    def get(self, bucket, key):
        """Download from 'S3'."""
        path = self.base_dir / bucket / key
        if path.suffix == '.csv':
            return pd.read_csv(path)
        return path.read_text()
    
    def list(self, bucket, prefix=''):
        """List objects in 'S3'."""
        bucket_path = self.base_dir / bucket
        if not bucket_path.exists():
            return []
        pattern = f"{prefix}**/*" if prefix else "**/*"
        return [str(p.relative_to(bucket_path)) for p in bucket_path.glob(pattern) if p.is_file()]

# Use the local simulator
sim = LocalS3Simulator()

# Create sample data
df = pd.DataFrame({
    'feature_1': np.random.randn(100),
    'feature_2': np.random.rand(100),
    'label': np.random.randint(0, 2, 100),
})

# Simulate the ML data pipeline
sim.put('my-ml-project', 'raw/features.csv', df)
sim.put('my-ml-project', 'processed/features_clean.csv', df.dropna())

# List objects
print("\nObjects in bucket:")
for obj in sim.list('my-ml-project'):
    print(f"  s3://my-ml-project/{obj}")

# Download and verify
downloaded = sim.get('my-ml-project', 'raw/features.csv')
print(f"\nDownloaded shape: {downloaded.shape}")

In [ ]:
# S3 client
s3 = boto3.client('s3', region_name=AWS_REGION)

# Helper functions for common S3 operations
def list_buckets():
    """List all S3 buckets in this account."""
    try:
        response = s3.list_buckets()
        buckets = [b['Name'] for b in response['Buckets']]
        print(f"Found {len(buckets)} bucket(s):")
        for b in buckets:
            print(f"  - {b}")
        return buckets
    except Exception as e:
        print(f"Error: {e}")
        return []

def create_bucket(bucket_name, region=AWS_REGION):
    """Create an S3 bucket."""
    try:
        if region == 'us-east-1':
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={'LocationConstraint': region}
            )
        print(f"Created bucket: {bucket_name}")
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f"Bucket already exists: {bucket_name}")
    except Exception as e:
        print(f"Error: {e}")

def upload_dataframe(df, bucket, key):
    """Upload a pandas DataFrame to S3 as CSV."""
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
    print(f"Uploaded {len(df)} rows to s3://{bucket}/{key}")

def download_dataframe(bucket, key):
    """Download a CSV from S3 into a pandas DataFrame."""
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.StringIO(response['Body'].read().decode('utf-8')))

# Demo (will run if AWS is configured)
buckets = list_buckets()

---
## Section 3: S3 — The ML Data Lake

S3 (Simple Storage Service) is the backbone of data storage on AWS.

### Core Concepts
- **Bucket**: Top-level container (globally unique name)
- **Key**: Full path to an object (e.g., `data/train/features.csv`)
- **Object**: Any file (CSV, model weights, images, logs)
- **Prefix**: Like a folder (e.g., `data/train/`)

### Naming Convention for ML Projects
```
your-ml-project/
├── raw/                    # Original data, never modified
│   └── 2024-01/
├── processed/              # Cleaned, feature-engineered data
│   └── features_v1.parquet
├── models/                 # Saved model artifacts
│   └── churn-model/v1.pkl
└── experiments/            # MLflow artifacts, logs
    └── run-abc123/
```

In [ ]:
# Check who you are (uses credentials from ~/.aws/credentials or env vars)
try:
    sts = boto3.client('sts', region_name=AWS_REGION)
    identity = sts.get_caller_identity()
    print(f"Account ID: {identity['Account']}")
    print(f"ARN:        {identity['Arn']}")
    print(f"User ID:    {identity['UserId']}")
except Exception as e:
    print(f"AWS not configured yet: {e}")
    print("\nTo configure:")
    print("  Option 1: aws configure (installs credentials to ~/.aws/)")
    print("  Option 2: Set environment variables:")
    print("    AWS_ACCESS_KEY_ID=...")
    print("    AWS_SECRET_ACCESS_KEY=...")
    print("    AWS_DEFAULT_REGION=us-east-1")

---
## Section 2: IAM — Identity & Access Management

IAM controls WHAT actions a user/service can perform on WHICH resources.

### Key Concepts
- **User**: A human (or CI/CD bot) with long-term credentials
- **Role**: Assumed by services (EC2, Lambda, SageMaker) — no long-term credentials
- **Policy**: JSON document defining allowed/denied actions
- **Least Privilege**: Grant ONLY what's needed

### Best Practices for ML Engineers
1. Never use root credentials for daily work
2. Create IAM users with specific policies (AmazonS3FullAccess, SageMakerFullAccess)
3. Use roles for SageMaker training jobs — they assume the role, not you
4. Use environment variables or AWS profiles — NEVER hardcode credentials

---
## Section 1: AWS Overview for ML Engineers

### The Core Services You'll Use Daily

| Service | What it does | ML Use Case |
|---------|-------------|-------------|
| **S3** | Object storage (files, datasets, models) | Store training data, model artifacts |
| **IAM** | Identity & Access Management | Control who can access what |
| **Lambda** | Serverless functions | Trigger training jobs, lightweight inference |
| **SageMaker** | Managed ML platform | Training, tuning, deployment |
| **ECR** | Container registry | Store Docker images for training/serving |
| **CloudWatch** | Logging & monitoring | Track model metrics, set alerts |

### Key Mental Model: Everything is an API

AWS services communicate via APIs. `boto3` is the Python SDK for these APIs.
Each service has a **client** (low-level API calls) and a **resource** (higher-level OO interface).

In [ ]:
import boto3
import json
import os
import io
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load credentials from .env
from dotenv import load_dotenv
load_dotenv('../../../.env')

AWS_REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
print(f"Working region: {AWS_REGION}")
print("boto3 version:", boto3.__version__)

# 01: AWS & Boto3 Fundamentals

## Learning Objectives
- Understand AWS core services for ML (S3, IAM, Lambda)
- Use boto3 to interact with AWS programmatically
- Build data pipelines using S3
- Understand IAM permissions and best practices

## Roadmap
IAM (who can do what) → S3 (data storage) → Lambda (serverless compute) → Putting it together